# Modelos Desafiantes

A ideia da modelagem inicial é obter insights do problema, além de sair com um modelo para trabalhar. A partir da análise exploratória, optei por iniciar a modelagem com o XGBoost que encaixa com esses objetivos e as características do problema:

* Lida com variáveis categóricas, valores faltantes e escalas variadas. Características que facilitam a prototipação rápida e identificação de variáveis relevantes.

* [Desempenha bem](https://arxiv.org/abs/2207.08815) em contexto de dados tabulares.

* É possível otimizar para geração de rankings [usando LambdaMART](https://xgboost.readthedocs.io/en/latest/tutorials/learning_to_rank.html).

* É fácil adicionar peso das amostras, possibilitando a otimização do modelo para métricas de negócio como receita ou rentabilidade.

* Fácil de integrar com valores SHAP para obtenção de insights.

* São modelos com implementações robustas para implantação em ambiente produtivo.



In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import optuna
from optuna import Study
from sklearn.metrics import roc_auc_score
import xgboost as xgb
from functools import partial
from validador import (
    validar_modelo,
    validar_modelo_por_segmento,
    gerar_relatorio,
    validar_modelo_por_tempo_relacionamento,
)

/home/gdarruda/Projects/case-sistema-recomendacao-resolucao/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Preparação dos dados

A classe `PreProcessador` é utilizada para separar os conjuntos de dados para treinamento, teste e validação. Eu utilizei uma separação temporal simples no método `separa_grupos`, usando os últimos 4 meses para validação e teste, sendo os 2 primeiros para validação e os 2 últimos para teste.

Eu criei o método `get_interacoes` para adicionar duas informações ao classificador: 

* receita que a contração trouxe usando escala logarítmica;

* flag binária identificando se o cliente clicou no produto em alguma interação anterior.

Por fim, temos também o método `get_popularidade_produto` para identificar os produtos mais contratados a partir das interações.

In [2]:
class PreProcessador:
    @staticmethod
    def separa_grupos(
        df: pd.DataFrame,
    ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

        safras_validacao = [202509, 202510]
        safras_teste = [202511, 202512]

        df_treino = df[~df["safra"].isin(safras_teste + safras_validacao)]
        df_validacao = df[df["safra"].isin(safras_validacao)]
        df_teste = df[df["safra"].isin(safras_teste)]

        return df_treino, df_validacao, df_teste

    @staticmethod
    def separa_grupos_random_ultimos_meses(
        df: pd.DataFrame,
        n_meses_ultimos: int = 4,
        frac_validacao: float = 0.5,
        random_state: int = 42,
    ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        safra_ordenada = sorted(df["safra"].unique())
        ultimas_safras = safra_ordenada[-n_meses_ultimos:]

        df_ultimos = df[df["safra"].isin(ultimas_safras)]
        df_treino = df[~df["safra"].isin(ultimas_safras)]

        df_validacao = df_ultimos.sample(frac=frac_validacao, random_state=random_state)
        df_teste = df_ultimos.drop(df_validacao.index)

        return df_treino, df_validacao, df_teste

    @staticmethod
    def get_interacoes(
        df_interacoes: pd.DataFrame, df_produtos: pd.DataFrame
    ) -> pd.DataFrame:

        df_interacoes = df_interacoes.copy()

        # Traduz a informação de receita em peso, para otimizações baseadas em custo
        df_interacoes["peso_receita"] = pd.merge(
            df_interacoes[["produto", "contratou"]],
            df_produtos[["produto", "receita_media"]],
            on="produto",
            how="left",
        ).assign(
            pesos_receita=lambda df: df["receita_media"].where(
                df["contratou"] == 1, 1.0
            )
        )["pesos_receita"]

        # Criar variável para identificar se o cliente já clicou no produto em interações anteriores
        df_interacoes = df_interacoes.sort_values(["id_cliente", "timestamp"])
        df_interacoes["clicou_produto_anteriormente"] = (
            df_interacoes.groupby(["id_cliente", "produto"])["clicou"]
            .cumsum()
            .groupby([df_interacoes["id_cliente"], df_interacoes["produto"]])
            .shift(1)
            .fillna(0)
            .apply(lambda x: 1 if x > 0 else 0)
        )

        return df_interacoes

    @staticmethod
    def get_popularidade_produto(
        df_clientes: pd.DataFrame,
        df_interacoes: pd.DataFrame,
        df_produtos: pd.DataFrame,
    ) -> pd.DataFrame:
        df_merged = pd.merge(
            df_interacoes,
            df_clientes[["id_cliente", "segmento"]],
            on="id_cliente",
            how="left",
        )

        df_contratacoes = df_merged[df_merged["contratou"] == 1].copy()

        popularidade = (
            df_contratacoes.groupby(["segmento", "produto"])
            .size()
            .reset_index(name="total_contratacoes")
        )

        popularidade = popularidade.sort_values(
            by=["segmento", "total_contratacoes"], ascending=[True, False]
        )

        return (
            pd.merge(
                pd.merge(
                    df_produtos[["produto"]],
                    pd.DataFrame({"segmento": ["basico", "intermediario", "premium"]}),
                    how="cross",
                ),
                popularidade,
                on=["segmento", "produto"],
                how="left",
            )
            .fillna(0)
            .pivot(index="segmento", values="total_contratacoes", columns="produto")
            .add_suffix("_vendas_segmento")
        )

A classe `DataprepXGBoost` é responsável por adaptar os dados para o modelo, destacando as principais transformações:

* Criar uma variável de cesta de produtos dos clientes, possibilitando o modelo identificar co-ocorrências de produtos.

* Transformar o segmento em variável numérica para criar uma relação de ordem entre segmentos.

* Popularidade dos produtos por segmento nas interações.

In [3]:
class DataprepXGBoost:
    def __init__(
        self,
        df_clientes: pd.DataFrame,
        df_interacoes: pd.DataFrame,
        df_contratos: pd.DataFrame,
        df_produtos: pd.DataFrame,
        df_popularidade_produtos: pd.DataFrame,
    ):
        self.df_clientes = df_clientes
        self.df_contratos = df_contratos
        self.df_interacoes = df_interacoes
        self.df_produtos = df_produtos
        self.df_popularidade_produtos = df_popularidade_produtos

    def _transforma_variaveis_clientes(self) -> pd.DataFrame:
        return (
            self.df_clientes.assign(
                segmento=lambda df: (
                    df["segmento"]
                    .replace({"basico": 1, "intermediario": 2, "premium": 3})
                    .astype(np.uint8)
                )
            )
            .join(
                pd.get_dummies(
                    self.df_clientes["canal_preferencial"],
                    dtype=np.uint8,
                    prefix="canal_preferencial",
                    prefix_sep="_",
                )
            )
            .join(
                pd.get_dummies(
                    self.df_clientes["genero"],
                    dtype=np.uint8,
                    prefix="genero",
                    prefix_sep="_",
                )
            )
            .drop(columns=["canal_preferencial", "uf", "genero"])
        )

    def _cria_cesta_clientes(self, data_limite: str) -> pd.DataFrame:

        df_contratos_filtrada = self.df_contratos.query(
            f"data_contratacao <= '{data_limite}'"
        )

        df_cesta = (
            pd.crosstab(
                df_contratos_filtrada["id_cliente"], df_contratos_filtrada["produto"]
            )
        ).astype(np.int8)

        return pd.merge(
            self.df_clientes[["id_cliente"]], df_cesta, on="id_cliente", how="left"
        ).fillna(0.0)

    def _transforma_popularidade_produtos(self) -> pd.DataFrame:
        return self.df_popularidade_produtos.reset_index().assign(
            segmento=lambda df: (
                df["segmento"]
                .replace({"basico": 1, "intermediario": 2, "premium": 3})
                .astype(np.uint8)
            )
        )

    def prepara_clientes(self) -> pd.DataFrame:

        return pd.merge(
            pd.merge(
                self._transforma_variaveis_clientes(),
                self._transforma_popularidade_produtos(),
                on="segmento",
            ),
            self._cria_cesta_clientes(
                datetime.strptime(str(self.df_interacoes.safra.max()), "%Y%m").strftime(
                    "%Y-%m-%d"
                ),
            ),
            how="inner",
            on="id_cliente",
        )

    def prepara_produtos(self):

        return self.df_produtos.join(
            pd.get_dummies(
                self.df_produtos["categoria"],
                dtype=np.uint8,
                prefix="categoria",
                prefix_sep="_",
            )
        ).drop(columns=["categoria", "publico_alvo"])

    def prepara_interacoes(
        self,
        filtra_cliente_contratantes,
    ):

        if filtra_cliente_contratantes:
            clientes_contratantes = (
                self.df_interacoes.groupby("id_cliente")[["contratou"]]
                .sum()
                .query("contratou > 0")
                .reset_index()
            )

            df_interacoes = pd.merge(
                self.df_interacoes,
                clientes_contratantes,
                on=["id_cliente"],
                how="inner",
                suffixes=("", "r"),
            )
        else:
            df_interacoes = self.df_interacoes

        return df_interacoes.assign(mes=lambda df: df["safra"] % 100)[
            [
                "id_cliente",
                "produto",
                "posicao_exibicao",
                "contratou",
                # "mes",
                "clicou_produto_anteriormente",
                "peso_receita",
            ]
        ]

    def prepara_tabela(
        self,
        filtra_cliente_contratantes: bool,
        colums_to_drop: list[str] = ["id_cliente", "contratou", "peso_receita"],
    ) -> tuple[pd.DataFrame, pd.Series, pd.Series]:

        df_interacoes_preparado = self.prepara_interacoes(filtra_cliente_contratantes)

        X = pd.merge(
            df_interacoes_preparado,
            self.prepara_clientes(),
            on="id_cliente",
            how="inner",
        ).drop(columns=colums_to_drop, errors="ignore")

        X = pd.merge(X, self.prepara_produtos(), on="produto", how="inner")
        X = X.assign(produto=lambda df: df["produto"].astype("category"))
        y = df_interacoes_preparado["contratou"]
        pesos = df_interacoes_preparado["peso_receita"]

        return X, y, pesos

    def criar_variaveis_predicao(self) -> pd.DataFrame:

        df_interacoes_cross = pd.merge(
            self.df_clientes["id_cliente"],
            pd.DataFrame({"produto": self.df_produtos["produto"].unique()}).assign(
                posicao_exibicao=1, contratou=0, peso_receita=1.0, safra=202601
            ),
            how="cross",
        )

        df_interacoes_cross = pd.merge(
            df_interacoes_cross,
            self.df_interacoes[
                ["id_cliente", "produto", "clicou_produto_anteriormente"]
            ]
            .groupby(by=["id_cliente", "produto"])
            .max()
            .reset_index(),
            on=["id_cliente", "produto"],
            how="left",
        ).assign(
            clicou_produto_anteriormente=lambda df: df[
                "clicou_produto_anteriormente"
            ].fillna(0)
        )

        return df_interacoes_cross

# Classificador "point wise" para recomendação

A classe do XGBoost é responsável por treinar os modelos com os pesos pré-determinados ou defini-los a partir da otimização com [Optuna](https://optuna.org). Os parâmetros definidos pelos atributos `melhores_parametros_default` e `melhores_parametros_ponderado` foram gerados por esse método de otimização. Eles foram separaddos para podermos ter modelos focados em métricas de negócios específicos, como focar em receita ao invés de contrações. 

Para além dos parâmetros "padrão" variados na otimização, gostaria de destacar alguns que considero importantes para o problema específico:

* subsample: fração das interações usada para treinar cada árvore. A ideia é evitar overfitting em um conjunto bem desbalanceado.

* colsample_bytree e colsample_bylevel: controlam a fração de variáveis (features) sorteadas para cada árvore ou nível, obrigando o modelo a aprender com diferentes combinações de dados. Pela análise exploratória, produtos diferentes se beneficiam de variáveis diferentes.

* max_bin: o tamanho dos bins do histograma para variáveis com alta variação, como saldo_medio_conta, um max_bin maior pode capturar detalhes importantes que impactam o NDCG@5.

* scale_pos_weight: contratações são eventos muito menos frequentes que as visualizações nas interações, esse peso ajuda o modelo a não ignorar os casos onde o cliente realmente contratou o produto.

A função `predizer_cliente` é uma forma de encapsular a predição como forma de facilitar os testes, gerando o ranking completo para todos os produtos. O produto entra como uma variável do modelo e a posição de recomendação é sempre a primeira, ao final é gerado um ranking com todos os produtos disponíveis e seu respectivo score.

In [4]:
class ClassificadorXGBoost:
    def __init__(self):
        self.study: None | Study = None
        self.modelo: None | xgb.XGBClassifier = None

        self.melhores_parametros_default = {
            "n_estimators": 700,
            "eta": 0.026294788236436252,
            "max_depth": 8,
            "min_child_weight": 10,
            "gamma": 2.768782597015123,
            "subsample": 0.9663847180469152,
            "colsample_bytree": 0.9528912074617883,
            "lambda": 5.704609857220126,
            "alpha": 0.7271468024912541,
            "scale_pos_weight": 1.00955441046317,
            "grow_policy": "depthwise",
            "colsample_bylevel": 0.5581652434609172,
            "max_bin": 457,
        }

        self.melhores_parametros_ponderado = {
            "n_estimators": 800,
            "eta": 0.012805040335549429,
            "max_depth": 7,
            "min_child_weight": 10,
            "gamma": 2.281640806992836,
            "subsample": 0.5118054941261478,
            "colsample_bytree": 0.7457283445514319,
            "lambda": 1.153746995748862,
            "alpha": 0.42047861588025537,
            "scale_pos_weight": 1.0644324592486043,
            "grow_policy": "depthwise",
            "colsample_bylevel": 0.9819207945395958,
            "max_bin": 309,
        }

    def executar_optuna(
        self,
        X_treino: pd.DataFrame,
        y_treino: pd.DataFrame,
        pesos_amostra_treino: None | pd.DataFrame,
        X_validacao: pd.DataFrame,
        y_validacao: pd.DataFrame,
        n_trials: int,
        pesos_amostra_validacao: None | list[float],
    ) -> Study:

        def objective(trial):
            params = {
                "objective": "binary:logistic",
                "enable_categorical": True,
                "random_state": 42,
                "n_jobs": -1,
                "eval_metric": "auc",
                "device": "cuda",
                "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=100),
                "eta": trial.suggest_float("eta", 0.01, 0.3, log=True),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "gamma": trial.suggest_float("gamma", 0.0, 5.0),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
                "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
                "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 20),
                "grow_policy": trial.suggest_categorical(
                    "grow_policy", ["depthwise", "lossguide"]
                ),
                "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
                "tree_method": "hist",
                "max_bin": trial.suggest_int("max_bin", 256, 1024),
            }

            sample_weight_eval_set = (
                [pesos_amostra_validacao] if pesos_amostra_validacao else None
            )

            clf = xgb.XGBClassifier(**params)
            clf.fit(
                X_treino,
                y_treino,
                sample_weight=pesos_amostra_treino,
                eval_set=[(X_validacao, y_validacao)],
                sample_weight_eval_set=sample_weight_eval_set,
                verbose=False,
            )

            y_pred = clf.predict_proba(X_validacao)[:, 1]
            return roc_auc_score(y_validacao, y_pred)

        self.study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=42),
        )

        self.study.optimize(objective, n_trials=n_trials, n_jobs=1)
        print("Best AUC:", self.study.best_value)
        print("Best params:", self.study.best_params)

        return self.study

    def treinar(
        self,
        X_treino: pd.DataFrame,
        y_treino: pd.Series,
        pesos_amostra: None | list[float],
    ):

        if self.study:
            melhores_parametros = self.study.best_params.copy()
        else:
            if pesos_amostra is None:
                melhores_parametros = self.melhores_parametros_default
            else:
                melhores_parametros = self.melhores_parametros_ponderado

        melhores_parametros.update(
            {
                "objective": "binary:logistic",
                "enable_categorical": True,
                "random_state": 42,
                "n_jobs": -1,
                "device": "cuda",
            }
        )
        self.modelo = xgb.XGBClassifier(**melhores_parametros)
        self.modelo.fit(X_treino, y_treino, sample_weight=pesos_amostra)

    def predizer_cliente(
        self,
        id_cliente: str,
        timestamp: str,
        X: pd.DataFrame,
        df_contratos: pd.DataFrame,
        df_produtos: pd.DataFrame,
    ) -> pd.DataFrame:

        assert self.modelo is not None, "É necessário treinar o modelo!"

        produtos_ativos_cliente = set(
            df_contratos[
                (df_contratos["id_cliente"] == id_cliente)
                & (df_contratos["status"] == "ativo")
                & (df_contratos["data_contratacao"] < timestamp[:10])
            ]["produto"].unique()
        )

        produtos_elegiveis_cliente = [
            produto
            for produto in df_produtos["produto"].unique().tolist()
            if produto not in produtos_ativos_cliente
        ]

        X_cliente = X[
            (X["id_cliente"] == id_cliente)
            & (X["produto"].isin(produtos_elegiveis_cliente))
        ].drop(columns=["id_cliente"])

        # X_cliente.insert(2, "mes", int(timestamp[5:7]))

        y = self.modelo.predict_proba(X_cliente)[:, 1]

        ranking = (
            X_cliente[["produto"]]
            .assign(score=y)
            .sort_values(by="score", ascending=False)
            .reset_index(drop=True)
        )

        ranking["posicao"] = ranking.index + 1

        return ranking

In [5]:
df_clientes = pd.read_csv("data/clientes.csv")
df_interacoes = pd.read_csv("data/interacoes.csv")
df_contratos = pd.read_csv("data/contratos_ativos.csv")
df_produtos = pd.read_csv("data/produtos.csv")

df_interacoes_processada = PreProcessador.get_interacoes(df_interacoes, df_produtos)

df_popularidade_produtos = PreProcessador.get_popularidade_produto(
    df_clientes, df_interacoes_processada, df_produtos
)

df_interacoes_treino, df_interacoes_validacao, df_interacoes_teste = (
    PreProcessador.separa_grupos(df_interacoes_processada)
)

data_prep_treino = DataprepXGBoost(
    df_clientes,
    df_interacoes_treino,
    df_contratos,
    df_produtos,
    df_popularidade_produtos,
)

X_treino, y_treino, pesos_amostra_treino = data_prep_treino.prepara_tabela(
    filtra_cliente_contratantes=False
)

data_prep_validacao = DataprepXGBoost(
    df_clientes,
    df_interacoes_treino,
    df_contratos,
    df_produtos,
    df_popularidade_produtos,
)

X_validacao, y_validacao, pesos_amostra_validacao = data_prep_validacao.prepara_tabela(
    filtra_cliente_contratantes=False
)

data_prep_completo = DataprepXGBoost(
    df_clientes,
    df_interacoes_processada,
    df_contratos,
    df_produtos,
    df_popularidade_produtos,
)

df_interacoes_cross = data_prep_completo.criar_variaveis_predicao()
data_prep_completo.df_interacoes = df_interacoes_cross
X_completo, _, _ = data_prep_completo.prepara_tabela(
    filtra_cliente_contratantes=False,
    colums_to_drop=["contratou", "peso_receita", "mes"],
)

In [6]:
df_interacoes_teste = pd.merge(
    df_interacoes_teste,
    df_clientes[["id_cliente", "segmento", "qtd_meses_cliente"]],
    on="id_cliente",
)

relatorios = []

In [7]:
classificador_xgb = ClassificadorXGBoost()

# classificador_xgb.executar_optuna(
#     X_treino, y_treino, None, X_validacao, y_validacao, 500, None
# )

classificador_xgb.treinar(X_treino, y_treino, None)

predizer_cliente_partial = partial(
    classificador_xgb.predizer_cliente,
    X=X_completo,
    df_contratos=df_contratos,
    df_produtos=df_produtos,
)

relatorio_xgb_classificador = gerar_relatorio(
    validar_modelo(df_interacoes_teste, predizer_cliente_partial),
    validar_modelo_por_segmento(df_interacoes_teste, predizer_cliente_partial),
    validar_modelo_por_tempo_relacionamento(
        df_interacoes_teste, predizer_cliente_partial
    ),
    "XGBoost Classificador",
)

relatorios.append(relatorio_xgb_classificador)
relatorio_xgb_classificador

Iniciando validação completa para 39906 casos de teste...


71it [00:00, 348.29it/s]/home/gdarruda/Projects/case-sistema-recomendacao-resolucao/.venv/lib/python3.14/site-packages/xgboost/core.py:751: UserWarning: [19:26:28] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
39906it [02:14, 297.09it/s]


Validando segmento: basico
Iniciando validação completa para 22067 casos de teste...


22067it [00:52, 420.68it/s]


Validando segmento: intermediario
Iniciando validação completa para 11931 casos de teste...


...

Validando clientes com meses de leracionamento: 6 a 12 meses
Iniciando validação completa para 5361 casos de teste...


5361it [00:07, 690.15it/s]


Validando clientes com meses de leracionamento: 3 a 6 meses
Iniciando validação completa para 3380 casos de teste...


3380it [00:04, 823.98it/s]


,Métricas Primárias.Precision@5,Métricas Primárias.NDCG@5 (Binário),Métricas Secundárias.Recall@5,Métricas Secundárias.Hit Rate@5,Métricas Secundárias.MAP@5,Métricas Secundárias.Catalog Coverage,Métricas de Receita.NDCG@5 (Receita),Métricas de Receita.Avg Revenue@5,Métricas de Receita.Revenue Recall@5,grupo,modelo
0,0.187115,0.671351,0.935574,0.935574,0.583987,0.60,0.671351,100.627367,0.935574,geral,XGBoost Classificador
1,0.194792,0.714072,0.973958,0.973958,0.627083,0.45,0.714072,106.420469,0.973958,basico,XGBoost Classificador
2,0.169412,0.627405,0.847059,0.847059,0.555490,0.35,0.627405,84.024000,0.847059,intermediario,XGBoost Classificador
3,0.187500,0.615514,0.937500,0.937500,0.510833,0.45,0.615514,104.365000,0.937500,premium,XGBoost Classificador
4,0.187097,0.650226,0.935484,0.935484,0.554839,0.40,0.650226,102.682903,0.935484,<3 meses,XGBoost Classificador
5,0.181818,0.669700,0.909091,0.909091,0.590530,0.60,0.669700,85.349318,0.909091,6 a 12 meses,XGBoost Classificador
6,0.190000,0.694460,0.950000,0.950000,0.611667,0.55,0.694460,118.824000,0.950000,3 a 6 meses,XGBoost Classificador


In [8]:
classificador_xgb = ClassificadorXGBoost()

# classificador_xgb.executar_optuna(
#     X_treino, y_treino, None, X_validacao, y_validacao, 500, None
# )

classificador_xgb.treinar(X_treino, y_treino, pesos_amostra_treino)

predizer_cliente_partial = partial(
    classificador_xgb.predizer_cliente,
    X=X_completo,
    df_contratos=df_contratos,
    df_produtos=df_produtos,
)

resultados_xgb_classificador_peso = gerar_relatorio(
    validar_modelo(df_interacoes_teste, predizer_cliente_partial),
    validar_modelo_por_segmento(df_interacoes_teste, predizer_cliente_partial),
    validar_modelo_por_tempo_relacionamento(
        df_interacoes_teste, predizer_cliente_partial
    ),
    "XGBoost Classificador (ponderado)",
)

relatorios.append(resultados_xgb_classificador_peso)
resultados_xgb_classificador_peso

Iniciando validação completa para 39906 casos de teste...


39906it [02:15, 294.87it/s]


Validando segmento: basico
Iniciando validação completa para 22067 casos de teste...


22067it [00:53, 413.45it/s]


Validando segmento: intermediario
Iniciando validação completa para 11931 casos de teste...


...

Validando clientes com meses de leracionamento: 6 a 12 meses
Iniciando validação completa para 5361 casos de teste...


5361it [00:07, 680.52it/s]


Validando clientes com meses de leracionamento: 3 a 6 meses
Iniciando validação completa para 3380 casos de teste...


3380it [00:04, 812.81it/s]


,Métricas Primárias.Precision@5,Métricas Primárias.NDCG@5 (Binário),Métricas Secundárias.Recall@5,Métricas Secundárias.Hit Rate@5,Métricas Secundárias.MAP@5,Métricas Secundárias.Catalog Coverage,Métricas de Receita.NDCG@5 (Receita),Métricas de Receita.Avg Revenue@5,Métricas de Receita.Revenue Recall@5,grupo,modelo
0,0.177591,0.638392,0.887955,0.887955,0.555229,0.70,0.638392,99.299748,0.887955,geral,XGBoost Classificador (ponderado)
1,0.184375,0.680509,0.921875,0.921875,0.600260,0.55,0.680509,104.772708,0.921875,basico,XGBoost Classificador (ponderado)
2,0.150588,0.539288,0.752941,0.752941,0.467843,0.55,0.539288,79.528706,0.752941,intermediario,XGBoost Classificador (ponderado)
3,0.190000,0.642611,0.950000,0.950000,0.540000,0.50,0.642611,107.171375,0.950000,premium,XGBoost Classificador (ponderado)
4,0.174194,0.612579,0.870968,0.870968,0.526344,0.55,0.612579,101.038710,0.870968,<3 meses,XGBoost Classificador (ponderado)
5,0.168182,0.576934,0.840909,0.840909,0.489015,0.65,0.576934,82.432727,0.840909,6 a 12 meses,XGBoost Classificador (ponderado)
6,0.180000,0.705775,0.900000,0.900000,0.641667,0.65,0.705775,117.096000,0.900000,3 a 6 meses,XGBoost Classificador (ponderado)


# Recomendação com Learning to Rank

O modelo de classificação (point-wise) proposto foca na propensão individual de cada produto, falhando em capturar a dinâmica de ranking necessária para um carrossel onde apenas as 5 primeiras posições são visíveis. Não é possível otimizar um modelo de gradient boosting diretamente com NDCG, mas o [LambdaMART](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/MSR-TR-2010-82.pdf) oferece uma solução calculando os valores λ. Esses valores funcionam como "gradiente substitutos", que representa a força e direção em que determinado item deve ser empurrado na lista, sendo uma otimização que está correlacionado com o NDCG. Essa abordagem correlaciona o treinamento diretamente com o ganho de NDCG, garantindo que os produtos de maior valor e probabilidade de contratação ocupem o topo da prateleira.

Um dos problemas para o uso dessa estratégia nesse problema é a raridade das contratações. Nese cenário, há pouca informação para fazermos as comparação binárias entre produtos, necessária para cálculo dos valores λ. Os dados foram filtrados para considerar apenas casos em que houve contratação, porque esse cenário não gera nenhuma informação (NDCG zerado). Entretanto, isso significa desconsiderar grande parcela dos dados.

E, talvez o mais crítico, é que o conjunto de dados não tem o conceito de "consulta" utilizado pelo modelo. A estratégia foi utilizar o cliente como uma "consulta", para que possamos comparar os produtos ofertados de alguma forma. O problema é que as ofertas foram feitas em contextos diferentes, um cliente pode simplesmente preferir usar um canal a outro e isso ser erroneamente transferido para o produto.

In [9]:
class ClassificadorXGBoostLTR:
    def __init__(self):
        self.study: None | Study = None
        self.modelo: None | xgb.XGBRanker = None
        self.id_cliente_map: dict = {}  # Mapa de id_cliente para índices qid

        self.melhores_parametros_default = {
            "objective": "rank:ndcg",
            "eval_metric": "ndcg",
            "n_estimators": 700,
            "learning_rate": 0.026294788236436252,
            "max_depth": 8,
            "min_child_weight": 10,
            "gamma": 2.768782597015123,
            "subsample": 0.9663847180469152,
            "colsample_bytree": 0.9528912074617883,
            "reg_lambda": 5.704609857220126,
            "reg_alpha": 0.7271468024912541,
            "grow_policy": "depthwise",
            "colsample_bylevel": 0.5581652434609172,
            "max_bin": 457,
            "tree_method": "hist",
            "enable_categorical": True,
            "random_state": 42,
        }

    def _preparar_grupos_qid(self, X: pd.DataFrame) -> tuple[list[int], dict]:

        clientes_unicos = X["id_cliente"].unique()
        mapa_id_cliente = {cliente: idx for idx, cliente in enumerate(clientes_unicos)}
        qid_array = X["id_cliente"].map(mapa_id_cliente).values.tolist()
        grupos = X.groupby("id_cliente").size().values.tolist()

        return qid_array, grupos

    def executar_optuna(
        self,
        X_treino: pd.DataFrame,
        y_treino: pd.Series,
        X_validacao: pd.DataFrame,
        y_validacao: pd.Series,
        n_trials: int = 50,
    ) -> Study:

        _, grupos_treino = self._preparar_grupos_qid(X_treino)
        _, grupos_validacao = self._preparar_grupos_qid(X_validacao)

        X_treino_feat = X_treino.drop(columns=["id_cliente"], errors="ignore")
        X_validacao_feat = X_validacao.drop(columns=["id_cliente"], errors="ignore")

        def objective(trial):
            params = {
                "objective": "rank:ndcg",
                "eval_metric": "ndcg",
                "enable_categorical": True,
                "random_state": 42,
                "n_jobs": -1,
                "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
                "eta": trial.suggest_float("eta", 0.01, 0.3, log=True),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "gamma": trial.suggest_float("gamma", 0.0, 5.0),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
                "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
                "grow_policy": trial.suggest_categorical(
                    "grow_policy", ["depthwise", "lossguide"]
                ),
                "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
                "tree_method": "hist",
                "max_bin": trial.suggest_int("max_bin", 256, 1024),
            }

            try:
                ranker = xgb.XGBRanker(**params)
                ranker.fit(
                    X_treino_feat,
                    y_treino,
                    group=grupos_treino,
                    eval_set=[(X_validacao_feat, y_validacao)],
                    eval_group=[grupos_validacao],
                    verbose=False,
                )

                results = ranker.evals_result()
                if results:
                    ndcg_scores = list(results.get("validation_0", {}).get("ndcg", []))
                    if ndcg_scores:
                        return ndcg_scores[-1]  # Retorna o score final

                return 0.0
            except Exception as e:
                print(f"Erro na trial: {e}")
                return 0.0

        self.study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=42),
        )

        self.study.optimize(
            objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True
        )
        print("Best NDCG:", self.study.best_value)
        print("Best params:", self.study.best_params)

        return self.study

    def treinar(
        self,
        X_treino: pd.DataFrame,
        y_treino: pd.Series,
    ):
        _, grupos_treino = self._preparar_grupos_qid(X_treino)

        X_treino_feat = X_treino.drop(columns=["id_cliente"], errors="ignore")

        if self.study:
            melhores_parametros = self.study.best_params.copy()
            melhores_parametros.update(
                {
                    "objective": "rank:ndcg",
                    "eval_metric": "ndcg",
                    "tree_method": "hist",
                    "enable_categorical": True,
                    "random_state": 42,
                }
            )
        else:
            melhores_parametros = self.melhores_parametros_default.copy()

        self.modelo = xgb.XGBRanker(**melhores_parametros)
        self.modelo.fit(
            X_treino_feat,
            y_treino,
            group=grupos_treino,
            verbose=False,
        )

    def predizer_cliente(
        self,
        id_cliente: str,
        timestamp: str,
        X: pd.DataFrame,
        df_contratos: pd.DataFrame,
        df_produtos: pd.DataFrame,
    ) -> pd.DataFrame:

        assert self.modelo is not None, "É necessário treinar o modelo!"

        produtos_ativos_cliente = set(
            df_contratos[
                (df_contratos["id_cliente"] == id_cliente)
                & (df_contratos["status"] == "ativo")
                & (df_contratos["data_contratacao"] < timestamp[:10])
            ]["produto"].unique()
        )

        produtos_elegiveis_cliente = [
            produto
            for produto in df_produtos["produto"].unique().tolist()
            if produto not in produtos_ativos_cliente
        ]

        X_cliente = X[
            (X["id_cliente"] == id_cliente)
            & (X["produto"].isin(produtos_elegiveis_cliente))
        ].drop(columns=["id_cliente"])

        X_cliente = X_cliente.copy()
        # X_cliente.insert(2, "mes", int(timestamp[5:7]))

        scores = self.modelo.predict(X_cliente)

        ranking = (
            X_cliente[["produto"]]
            .assign(score=scores)
            .sort_values(by="score", ascending=False)
            .reset_index(drop=True)
        )

        ranking["posicao"] = ranking.index + 1

        return ranking

In [10]:
X_treino_ltr, y_treino_ltr, _ = data_prep_treino.prepara_tabela(
    filtra_cliente_contratantes=True, colums_to_drop=["contratou", "peso_receita"]
)

X_validacao_ltr, y_validacao_ltr, _ = data_prep_validacao.prepara_tabela(
    filtra_cliente_contratantes=True, colums_to_drop=["contratou", "peso_receita"]
)

classificador_ltr = ClassificadorXGBoostLTR()
# classificador_ltr.executar_optuna(
#     X_treino_ltr,
#     y_treino_ltr,
#     X_validacao_ltr,
#     y_validacao_ltr,
#     n_trials=200,
# )

classificador_ltr.treinar(X_treino_ltr, y_treino_ltr)

predizer_cliente_partial = partial(
    classificador_ltr.predizer_cliente,
    X=X_completo,
    df_contratos=df_contratos,
    df_produtos=df_produtos,
)


resultados_xbg_ltr = gerar_relatorio(
    validar_modelo(df_interacoes_teste, predizer_cliente_partial),
    validar_modelo_por_segmento(df_interacoes_teste, predizer_cliente_partial),
    validar_modelo_por_tempo_relacionamento(
        df_interacoes_teste, predizer_cliente_partial
    ),
    "XGBoost LTR",
)

relatorios.append(resultados_xbg_ltr)
resultados_xbg_ltr

Iniciando validação completa para 39906 casos de teste...


39906it [02:18, 288.18it/s]


Validando segmento: basico
Iniciando validação completa para 22067 casos de teste...


22067it [00:51, 431.71it/s]


Validando segmento: intermediario
Iniciando validação completa para 11931 casos de teste...


...

Validando clientes com meses de leracionamento: 6 a 12 meses
Iniciando validação completa para 5361 casos de teste...


5361it [00:07, 738.11it/s]


Validando clientes com meses de leracionamento: 3 a 6 meses
Iniciando validação completa para 3380 casos de teste...


3380it [00:03, 859.36it/s]


,Métricas Primárias.Precision@5,Métricas Primárias.NDCG@5 (Binário),Métricas Secundárias.Recall@5,Métricas Secundárias.Hit Rate@5,Métricas Secundárias.MAP@5,Métricas Secundárias.Catalog Coverage,Métricas de Receita.NDCG@5 (Receita),Métricas de Receita.Avg Revenue@5,Métricas de Receita.Revenue Recall@5,grupo,modelo
0,0.182073,0.648954,0.910364,0.910364,0.562278,0.60,0.648954,98.056891,0.910364,geral,XGBoost LTR
1,0.193750,0.694550,0.968750,0.968750,0.602604,0.45,0.694550,106.006354,0.968750,basico,XGBoost LTR
2,0.167059,0.602733,0.835294,0.835294,0.526471,0.50,0.602733,85.515882,0.835294,intermediario,XGBoost LTR
3,0.170000,0.588632,0.850000,0.850000,0.503542,0.50,0.588632,92.303000,0.850000,premium,XGBoost LTR
4,0.187097,0.660718,0.935484,0.935484,0.569355,0.45,0.660718,103.541613,0.935484,<3 meses,XGBoost LTR
5,0.177273,0.667035,0.886364,0.886364,0.593561,0.55,0.667035,83.542273,0.886364,6 a 12 meses,XGBoost LTR
6,0.180000,0.690787,0.900000,0.900000,0.620833,0.55,0.690787,117.096000,0.900000,3 a 6 meses,XGBoost LTR


In [ ]:
df_relatorios = pd.concat(relatorios).reset_index(drop=True)
df_relatorios.to_excel("relatorios/resultados_desafiantes.xlsx")
df_relatorios

,Métricas Primárias.Precision@5,Métricas Primárias.NDCG@5 (Binário),Métricas Secundárias.Recall@5,Métricas Secundárias.Hit Rate@5,Métricas Secundárias.MAP@5,Métricas Secundárias.Catalog Coverage,Métricas de Receita.NDCG@5 (Receita),Métricas de Receita.Avg Revenue@5,Métricas de Receita.Revenue Recall@5,grupo,modelo
0,0.187115,0.671351,0.935574,0.935574,0.583987,0.60,0.671351,100.627367,0.935574,geral,XGBoost Classificador
1,0.194792,0.714072,0.973958,0.973958,0.627083,0.45,0.714072,106.420469,0.973958,basico,XGBoost Classificador
2,0.169412,0.627405,0.847059,0.847059,0.555490,0.35,0.627405,84.024000,0.847059,intermediario,XGBoost Classificador
3,0.187500,0.615514,0.937500,0.937500,0.510833,0.45,0.615514,104.365000,0.937500,premium,XGBoost Classificador
4,0.187097,0.650226,0.935484,0.935484,0.554839,0.40,0.650226,102.682903,0.935484,<3 meses,XGBoost Classificador
5,0.181818,0.669700,0.909091,0.909091,0.590530,0.60,0.669700,85.349318,0.909091,6 a 12 meses,XGBoost Classificador
6,0.190000,0.694460,0.950000,0.950000,0.611667,0.55,0.694460,118.824000,0.950000,3 a 6 meses,XGBoost Classificador
7,0.177591,0.638392,0.887955,0.887955,0.555229,0.70,0.638392,99.299748,0.887955,geral,XGBoost Classificador (ponderado)
8,0.184375,0.680509,0.921875,0.921875,0.600260,0.55,0.680509,104.772708,0.921875,basico,XGBoost Classificador (ponderado)
9,0.150588,0.539288,0.752941,0.752941,0.467843,0.55,0.539288,79.528706,0.752941,intermediario,XGBoost Classificador (ponderado)
